In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
df_mmlu = pd.read_csv("dataset/mmlu_original.csv")

In [6]:
df_mmlu.head()

,i,subject,question,choices,selected_choice,answer,label
0,0-correct,abstract_algebra,Find the degree for the given field extension ...,['0' '4' '2' '6'],1,4,correct
1,0-incorrect,abstract_algebra,Find the degree for the given field extension ...,['0' '4' '2' '6'],3,6,incorrect
2,1-correct,abstract_algebra,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",['8' '2' '24' '120'],2,24,correct
3,1-incorrect,abstract_algebra,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",['8' '2' '24' '120'],1,2,incorrect
4,2-correct,abstract_algebra,Find all zeros in the indicated finite field o...,"['0' '1' '0,1' '0,4']",3,"0,4",correct


In [7]:
df_athene = pd.read_csv("dataset/Athene-70B.csv")

In [8]:
df_athene.head()

,i,llm_judge,pred,label,logit_yes,logit_no
0,0,Nexusflow/Athene-70B,correct,correct,23.625,20.125
1,1,Nexusflow/Athene-70B,incorrect,incorrect,22.750,23.625
2,2,Nexusflow/Athene-70B,correct,correct,24.250,21.750
3,3,Nexusflow/Athene-70B,correct,incorrect,24.375,21.375
4,4,Nexusflow/Athene-70B,incorrect,correct,21.625,23.375


In [9]:
len(df_mmlu), len(df_athene)

(28084, 28084)

In [17]:
df_mmlu["i"].describe()

count         28084
unique        28084
top       0-correct
freq              1
Name: i, dtype: object

### Merging to get the complete view of all the datasets available

In [31]:
def create_master_csv(dataset_dir="dataset"):
    dataset_dir = Path(dataset_dir)

    original_path = dataset_dir / "mmlu_original.csv"
    master_path = dataset_dir / "master.csv"

    if not original_path.exists():
        raise FileNotFoundError(f"Missing original dataset: {original_path}")

    original = pd.read_csv(original_path, dtype={"i": str})

    if "i" not in original.columns or "label" not in original.columns:
        raise ValueError("Original dataset must contain 'i' and 'label'.")

    # The original row position is the shared ID for merging.
    original["_row_id"] = range(len(original))

    # Exclude generated files so they are never merged as judge inputs.
    excluded_files = {
        "mmlu_original.csv",
        "master.csv",
        "single_judge_merged.csv",
        "master_with_scores.csv",
    }

    prediction_files = [
        path for path in sorted(dataset_dir.glob("*.csv"))
        if path.name not in excluded_files
        and not path.stem.endswith("_merged")
    ]

    if not prediction_files:
        raise FileNotFoundError("No judge prediction CSV files found.")

    master = original.copy()
    loaded_judges = []

    print(f"Original rows: {len(original)}")
    print(f"Judge CSV files found: {len(prediction_files)}\n")

    for csv_path in prediction_files:
        judge_name = csv_path.stem

        print(f"Processing: {csv_path.name}")

        judge = pd.read_csv(csv_path, dtype={"i": str})

        required_columns = {
            "i", "label", "pred", "logit_yes", "logit_no"
        }

        missing = required_columns - set(judge.columns)

        if missing:
            print(f"  Skipped: missing columns {sorted(missing)}")
            continue

        # Convert judge row IDs to integers.
        row_ids = pd.to_numeric(judge["i"], errors="coerce")

        if row_ids.isna().any():
            raise ValueError(
                f"{csv_path.name}: non-numeric values in column 'i'."
            )

        if (row_ids % 1 != 0).any():
            raise ValueError(
                f"{csv_path.name}: non-integer row IDs detected."
            )

        judge["_row_id"] = row_ids.astype(int)

        # Validate unique IDs and exact coverage.
        expected_ids = set(range(len(original)))
        actual_ids = set(judge["_row_id"])

        if judge["_row_id"].duplicated().any():
            raise ValueError(
                f"{csv_path.name}: duplicate row IDs detected."
            )

        if actual_ids != expected_ids or len(judge) != len(original):
            raise ValueError(
                f"{csv_path.name}: row IDs do not match the original "
                f"dataset. Expected {len(original)} rows; got {len(judge)}."
            )

        # Sort by row ID to make the alignment explicit.
        judge = judge.sort_values("_row_id").reset_index(drop=True)

        # Confirm the judge's ground-truth label matches the original.
        original_labels = (
            original.sort_values("_row_id")["label"]
            .astype(str).str.strip().str.lower().reset_index(drop=True)
        )
        judge_labels = (
            judge["label"]
            .astype(str).str.strip().str.lower().reset_index(drop=True)
        )

        agreement = original_labels.eq(judge_labels)

        if not agreement.all():
            mismatch_count = int((~agreement).sum())
            raise ValueError(
                f"{csv_path.name}: {mismatch_count} ground-truth "
                "label mismatches. Merge stopped to prevent bad alignment."
            )

        # Prefix judge-specific columns to avoid collisions.
        judge = judge.rename(columns={
            column: f"{judge_name}_{column}"
            for column in judge.columns
            if column != "_row_id"
        })

        # Each judge contributes one row per original row.
        master = master.merge(
            judge,
            on="_row_id",
            how="left",
            validate="one_to_one"
        )

        # Keep an explicit per-judge label agreement indicator.
        master[f"{judge_name}_label_agrees"] = (
            master["label"].astype(str).str.strip().str.lower()
            == master[f"{judge_name}_label"]
                .astype(str).str.strip().str.lower()
        )

        loaded_judges.append(judge_name)

        print(f"  Rows: {len(judge)}")
        print("  Row IDs: valid")
        print("  Ground-truth labels: 100% agreement")
        print("  Merge: successful\n")

    if not loaded_judges:
        raise ValueError("No valid judge datasets were merged.")

    # Remove only the temporary join key.
    master.drop(columns=["_row_id"], inplace=True)

    # Ensure the master contains all original examples.
    if len(master) != len(original):
        raise ValueError(
            f"Unexpected row count: {len(master)} vs {len(original)}"
        )

    master.to_csv(master_path, index=False)

    print("=" * 60)
    print("Master CSV created successfully.")
    print(f"Judges merged: {len(loaded_judges)}")
    print(f"Rows: {len(master)}")
    print(f"Columns: {len(master.columns)}")
    print(f"Output: {master_path.resolve()}")
    print("Included judges:", loaded_judges)

    return master


In [32]:
master_df = create_master_csv()

Original rows: 28084
Judge CSV files found: 17

Processing: Athene-70B.csv
  Rows: 28084
  Row IDs: valid
  Ground-truth labels: 100% agreement
  Merge: successful

Processing: dolphin-2.1-mistral-7b.csv
  Rows: 28084
  Row IDs: valid
  Ground-truth labels: 100% agreement
  Merge: successful

Processing: dolphin-2.5-mixtral-8x7b.csv
  Rows: 28084
  Row IDs: valid
  Ground-truth labels: 100% agreement
  Merge: successful

Processing: Hermes-3-Llama-3.1-70B.csv
  Rows: 28084
  Row IDs: valid
  Ground-truth labels: 100% agreement
  Merge: successful

Processing: Meta-Llama-3-70B-Instruct.csv
  Rows: 28084
  Row IDs: valid
  Ground-truth labels: 100% agreement
  Merge: successful

Processing: Meta-Llama-3-8B-Instruct.csv
  Rows: 28084
  Row IDs: valid
  Ground-truth labels: 100% agreement
  Merge: successful

Processing: Meta-Llama-3-8B.csv
  Rows: 28084
  Row IDs: valid
  Ground-truth labels: 100% agreement
  Merge: successful

Processing: Mistral-7B-Instruct-v0.1.csv
  Rows: 28084
  Row 